In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.preprocess import (
    load_data,
    clean_data,
    handle_missing_values
)

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

In [2]:
df = load_data("../data/customer_churn_1M.csv")

df = clean_data(df)

df = handle_missing_values(df)

print(df.shape)

(1000000, 31)


In [3]:
target = "churn"

categorical_cols = (
    df.select_dtypes(include=["object"])
      .columns
      .tolist()
)

numerical_cols = (
    df.select_dtypes(include=["int64", "float64"])
      .columns
      .tolist()
)

numerical_cols.remove(target)

In [4]:
X = df.drop("churn", axis=1)

y = df["churn"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [7]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=15,
                min_samples_split=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [8]:
rf_model.fit(X_train, y_train)

print("Random Forest Training Complete")

Random Forest Training Complete


In [9]:
X_train.shape

(800000, 30)

In [13]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, rf_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.67      0.78    180155
           1       0.16      0.58      0.25     19845

    accuracy                           0.66    200000
   macro avg       0.55      0.63      0.52    200000
weighted avg       0.86      0.66      0.73    200000



In [12]:
cm = confusion_matrix(y_test, rf_pred)
print("confusion matrix:")
print(cm)

confusion matrix:
[[121346  58809]
 [  8372  11473]]


In [11]:
auc = roc_auc_score(y_test, rf_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.6769090405403051


In [14]:
import joblib

joblib.dump(
    rf_model,
    "../models/random_forest.pkl"
)

['../models/random_forest.pkl']

In [16]:
import joblib

rf_model = joblib.load(
    "../models/random_forest.pkl"
)
print("rf model feature importances:", rf_model.named_steps["classifier"].feature_importances_)

rf model feature importances: [0.03453702 0.04029607 0.01114702 0.02930439 0.00441918 0.04798716
 0.04427508 0.02174854 0.00441259 0.00238326 0.01231363 0.00397166
 0.00386888 0.02328683 0.00459249 0.00455361 0.09183656 0.04819296
 0.05331989 0.02406396 0.03631671 0.03641518 0.03986661 0.0422607
 0.00396356 0.00405158 0.00195267 0.00405175 0.00404752 0.00397306
 0.00386287 0.00302824 0.00379307 0.0041227  0.00414428 0.00293865
 0.0394956  0.08397619 0.14380422 0.00403071 0.0040204  0.00396579
 0.00388648 0.00373921 0.0037815 ]


In [17]:
preprocessor = rf_model.named_steps["preprocessor"]

feature_names = preprocessor.get_feature_names_out()

import pandas as pd

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.named_steps["classifier"].feature_importances_
})

fi = fi.sort_values(
    by="importance",
    ascending=False
)

fi.head(20)

,feature,importance
38,cat__contract_two_year,0.143804
16,num__customer_satisfaction,0.091837
37,cat__contract_one_year,0.083976
18,num__num_service_calls,0.053320
17,num__num_complaints,0.048193
5,num__monthlycharges,0.047987
6,num__totalcharges,0.044275
23,num__days_since_signup,0.042261
1,num__annual_income,0.040296
22,num__credit_score,0.039867
